In [ ]:
pip install ipykernel
python -m ipykernel install --user --name=myenv --display-name "dev-env"

In [1]:
# 0) Install minimal deps (idempotent)
# %pip install --quiet --no-cache-dir transformers peft accelerate datasets sentencepiece
# !pip install --upgrade pip
# !pip install --upgrade transformers accelerate datasets peft
# !pip install --upgrade bitsandbytes

In [2]:
# import transformers, accelerate, peft, datasets
# print("transformers:", transformers.__version__)
# print("accelerate:", accelerate.__version__) 
# print("peft:", peft.__version__)

# from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
# from peft import LoraConfig, get_peft_model, TaskType
# print("✅ imports OK")

In [5]:
# import os
# os.environ["TRANSFORMERS_NO_TORCHVISION"] = "1"
# %pip install --quiet --no-cache-dir "transformers==4.44.2" "accelerate==0.34.0" "peft==0.12.0" "datasets>=2.20.0" sentencepiece

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.23.0 requires accelerate>=1.4.0, but you have accelerate 0.34.0 which is incompatible.
trl 0.23.0 requires transformers>=4.56.1, but you have transformers 4.44.2 which is incompatible.
unsloth 2025.9.5 requires accelerate>=0.34.1, but you have accelerate 0.34.0 which is incompatible.
unsloth 2025.9.5 requires datasets<4.0.0,>=3.4.1, but you have datasets 4.1.1 which is incompatible.
unsloth 2025.9.5 requires transformers!=4.47.0,!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,>=4.51.3, but you have transformers 4.44.2 which is incompatible.
unsloth-zoo 2025.9.6 requires accelerate>=0.34.1, but you have accelerate 0.34.0 which is incompatible.
unsloth-zoo 2025.9.6 requires datasets<4.0.0,>=3.4.1, but you have datasets 4.1.1 which is incompatible.
unsloth-zoo 2025.9.6 requires transform

In [1]:
# fine_tune_demo.ipynb
# Step 1: Imports
import os
import json
from datasets import load_dataset, Dataset
import torch
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, TaskType
from transformers import pipeline

ModuleNotFoundError: Could not import module 'Trainer'. Are this object's requirements defined correctly?

In [2]:
# Step 2: Create a toy dataset in Alpaca-style format
train_data = [
    {
        "instruction": "Translate English to French",
        "input": "Hello",
        "output": "Bonjour"
    },
    {
        "instruction": "Translate English to French",
        "input": "Goodbye",
        "output": "Au revoir"
    },
    {
        "instruction": "What is 2+2?",
        "input": "",
        "output": "4"
    },
    {
        "instruction": "Who wrote Hamlet?",
        "input": "",
        "output": "William Shakespeare"
    }
]

In [3]:
os.makedirs("/workspace/data/processed", exist_ok=True)
jsonl_path = "/workspace/data/processed/train.jsonl"

with open(jsonl_path, "w") as f:
    for row in train_data:
        f.write(json.dumps(row) + "\n")

print("✅ Saved toy dataset to", jsonl_path)

✅ Saved toy dataset to /workspace/data/processed/train.jsonl


In [4]:
# Step 3: Load dataset
ds = load_dataset("json", data_files=jsonl_path, split="train")
print(ds)

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 4
})


In [10]:
# Step 4: Pick a tiny base model (super light)
BASE = "sshleifer/tiny-gpt2"

def fmt(ex):
    instr = ex.get("instruction","")
    inp   = ex.get("input","")
    out   = ex.get("output","")
    prompt = f"### Instruction:\n{instr}\n\n### Input:\n{inp}\n\n### Response:\n"
    return {"text": prompt + out, "prompt": prompt, "labels": out}

ds = ds.map(fmt)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [12]:
# 5) Tokenize with Transformers
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

MAX_LEN = 256
def tok_fn(ex):
    # We train to generate 'labels' after 'prompt'
    full = tok(ex["text"], truncation=True, max_length=MAX_LEN)
    # Mask prompt tokens so loss only applies to the response
    prompt_ids = tok(ex["prompt"], truncation=True, max_length=MAX_LEN)["input_ids"]
    labels = full["input_ids"][:]
    mask_len = min(len(prompt_ids), len(labels))
    labels[:mask_len] = [-100] * mask_len
    full["labels"] = labels
    return full


ds_tok = ds.map(tok_fn, remove_columns=ds.column_names)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [14]:
# 6) Build PEFT LoRA on tiny GPT-2
device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained(BASE).to(device)

lora = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=["c_attn","c_proj"],  # GPT-2 proj layers
)
model = get_peft_model(model, lora)

NameError: name 'AutoModelForCausalLM' is not defined

In [ ]:
# 7) Train briefly
OUT_DIR = "/workspace/adapters/myrun-demo"
os.makedirs(OUT_DIR, exist_ok=True)

args = TrainingArguments(
    output_dir=OUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=1,
    save_steps=50,
    fp16=torch.cuda.is_available(),
)
dcoll = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)
trainer = Trainer(model=model, args=args, train_dataset=ds_tok, data_collator=dcoll)

trainer.train()

In [ ]:
# 8) Save LoRA adapter + tokenizer
trainer.model.save_pretrained(OUT_DIR)
tok.save_pretrained(OUT_DIR)
print("✅ LoRA adapter saved to:", OUT_DIR)

# 9) Quick test (generate)
pipe = pipeline("text-generation", model=model, tokenizer=tok, device=0 if torch.cuda.is_available() else -1)
print(pipe("### Instruction:\nTranslate English to French\n\n### Input:\nHello\n\n### Response:\n", max_new_tokens=20)[0]["generated_text"])